# 01. Data Audit and Cleaning

**Team members:** _Add names here_

This notebook prepares the raw forest-fire dataset for analysis. It reconstructs the two regional blocks into one table, repairs the known malformed observation, standardizes text and data types, validates the cleaned dataset, and saves a single clean CSV for the next notebook.

Feature engineering and model-related preprocessing are intentionally left for later notebooks so that each stage has a clear responsibility.

## 1. Imports and Data Path

The analysis begins from the original `forest_fires_dataset.csv` file.

In [1]:
from pathlib import Path
import pandas as pd


data_path = Path("forest_fires_dataset.csv")
assert data_path.exists(), f"Dataset not found: {data_path.resolve()}"

## 2. Inspect the Raw File Structure

The source file is not a standard single-table CSV: it contains region labels, a repeated header, and structural rows. Inspecting the first lines verifies this layout before reconstruction.

In [2]:
with data_path.open("r", encoding="utf-8") as file:
    for line_number in range(15):
        line = file.readline()
        print(line_number + 1, repr(line))

1 'Cordillera,,,,,,,,,,,,,\n'
2 'day,month,year,Temperature, RH, Ws,Rain ,FFMC,DMC,DC,ISI,BUI,FWI,Classes  \n'
3 '1,6,2012,29,57,18,0,65.7,3.4,7.6,1.3,3.4,0.5,not fire   \n'
4 '2,6,2012,29,61,13,1.3,64.4,4.1,7.6,1,3.9,0.4,not fire   \n'
5 '3,6,2012,26,82,22,13.1,47.1,2.5,7.1,0.3,2.7,0.1,not fire   \n'
6 '4,6,2012,25,89,13,2.5,28.6,1.3,6.9,0,1.7,0,not fire   \n'
7 '5,6,2012,27,77,16,0,64.8,3,14.2,1.2,3.9,0.5,not fire   \n'
8 '6,6,2012,31,67,14,0,82.6,5.8,22.2,3.1,7,2.5,fire   \n'
9 '7,6,2012,33,54,13,0,88.2,9.9,30.5,6.4,10.9,7.2,fire   \n'
10 '8,6,2012,30,73,15,0,86.6,12.1,38.3,5.6,13.5,7.1,fire   \n'
11 '9,6,2012,25,88,13,0.2,52.9,7.9,38.8,0.4,10.5,0.3,not fire   \n'
12 '10,6,2012,28,79,12,0,73.2,9.5,46.3,1.3,12.6,0.9,not fire   \n'
13 '11,6,2012,31,65,14,0,84.5,12.5,54.3,4,15.8,5.6,fire   \n'
14 '12,6,2012,26,81,19,0,84,13.8,61.4,4.8,17.7,7.1,fire   \n'
15 '13,6,2012,27,84,21,1.2,50,6.7,17,0.5,6.7,0.2,not fire   \n'


In [3]:
df_raw = pd.read_csv(
    data_path,
    header=None,
    dtype=str
)

print("Raw shape:", df_raw.shape)
df_raw.head(10)

Raw shape: (249, 14)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,Cordillera,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,day,month,year,Temperature,RH,Ws,Rain,FFMC,DMC,DC,ISI,BUI,FWI,Classes
2,1,6,2012,29,57,18,0,65.7,3.4,7.6,1.3,3.4,0.5,not fire
3,2,6,2012,29,61,13,1.3,64.4,4.1,7.6,1,3.9,0.4,not fire
4,3,6,2012,26,82,22,13.1,47.1,2.5,7.1,0.3,2.7,0.1,not fire
5,4,6,2012,25,89,13,2.5,28.6,1.3,6.9,0,1.7,0,not fire
6,5,6,2012,27,77,16,0,64.8,3,14.2,1.2,3.9,0.5,not fire
7,6,6,2012,31,67,14,0,82.6,5.8,22.2,3.1,7,2.5,fire
8,7,6,2012,33,54,13,0,88.2,9.9,30.5,6.4,10.9,7.2,fire
9,8,6,2012,30,73,15,0,86.6,12.1,38.3,5.6,13.5,7.1,fire


## 3. Reconstruct the Observation Table

The rows labelled `Cordillera` and `Hudson Bay` identify the region for the observations that follow them. The region label is carried forward, and only rows whose first field is a numeric day are retained as observations.

In [4]:
region_names = ["Cordillera", "Hudson Bay"]
is_region_row = df_raw[0].str.strip().isin(region_names)

df_raw.loc[is_region_row]

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,Cordillera,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
125,Hudson Bay,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Assign each region marker to its following observations.
df_raw["Region"] = df_raw[0].where(is_region_row)
df_raw["Region"] = df_raw["Region"].ffill()

# Observation rows have a numeric day in the first column.
day_numeric = pd.to_numeric(
    df_raw[0].str.strip(),
    errors="coerce"
)

is_observation = day_numeric.notna()
df_clean = df_raw.loc[is_observation].copy()

print("Observation rows:", df_clean.shape[0])

Observation rows: 244


In [6]:
# The true column names are stored in the first repeated header row.
column_names = (
    df_raw.loc[1, list(range(14))]
    .str.strip()
    .tolist()
)

df_clean.columns = column_names + ["Region"]
df_clean.columns = df_clean.columns.str.strip()
df_clean.reset_index(drop=True, inplace=True)

df_clean.head()

,day,month,year,Temperature,RH,Ws,Rain,FFMC,DMC,DC,ISI,BUI,FWI,Classes,Region
0,1,6,2012,29,57,18,0,65.7,3.4,7.6,1.3,3.4,0.5,not fire,Cordillera
1,2,6,2012,29,61,13,1.3,64.4,4.1,7.6,1,3.9,0.4,not fire,Cordillera
2,3,6,2012,26,82,22,13.1,47.1,2.5,7.1,0.3,2.7,0.1,not fire,Cordillera
3,4,6,2012,25,89,13,2.5,28.6,1.3,6.9,0,1.7,0,not fire,Cordillera
4,5,6,2012,27,77,16,0,64.8,3,14.2,1.2,3.9,0.5,not fire,Cordillera


## 4. Repair the Known Malformed Observation

One observation in the raw file is missing a comma, which shifts the final fields to the right. The affected row is identified by its missing target value, and the verified values are repaired explicitly.

In [7]:
df_clean[df_clean["Classes"].isna()]

,day,month,year,Temperature,RH,Ws,Rain,FFMC,DMC,DC,ISI,BUI,FWI,Classes,Region
165,14,7,2012,37,37,18,0.2,88.9,12.9,14.6 9,12.5,10.4,fire,NaN,Hudson Bay


In [8]:
# Correct the known malformed row.
df_clean.loc[
    165,
    ["DC", "ISI", "BUI", "FWI", "Classes"]
] = ["14.6", "9", "12.5", "10.4", "fire"]

df_clean.loc[165]

day                    14
month                   7
year                 2012
Temperature            37
RH                     37
Ws                     18
Rain                  0.2
FFMC                 88.9
DMC                  12.9
DC                   14.6
ISI                     9
BUI                  12.5
FWI                  10.4
Classes              fire
Region         Hudson Bay
Name: 165, dtype: str

## 5. Standardize Text and Data Types

Whitespace is removed from the categorical labels, and the measurement/date columns are converted to numeric types. Using `errors="raise"` ensures that any remaining malformed numeric value stops execution instead of being silently converted to missing data.

In [9]:
# Clean categorical text.
df_clean["Classes"] = df_clean["Classes"].str.strip()
df_clean["Region"] = df_clean["Region"].str.strip()

numeric_columns = [
    "day", "month", "year", "Temperature", "RH", "Ws",
    "Rain", "FFMC", "DMC", "DC", "ISI", "BUI", "FWI"
]

df_clean[numeric_columns] = df_clean[numeric_columns].apply(
    pd.to_numeric,
    errors="raise"
)

df_clean.dtypes

day              int64
month            int64
year             int64
Temperature      int64
RH               int64
Ws               int64
Rain           float64
FFMC           float64
DMC            float64
DC             float64
ISI            float64
BUI            float64
FWI            float64
Classes            str
Region             str
dtype: object

## 6. Validate the Cleaned Dataset

The final audit checks the expected shape, missing values, duplicates, valid class/region labels, and valid calendar dates. The date conversion below is only a validation check; no derived date feature is saved during cleaning.

In [10]:
print("Shape:", df_clean.shape)
print("Missing values:", int(df_clean.isna().sum().sum()))
print("Duplicate rows:", int(df_clean.duplicated().sum()))

print()
print("Class counts:")
print(df_clean["Classes"].value_counts())

print()
print("Region counts:")
print(df_clean["Region"].value_counts())

# Confirm that year/month/day combinations are valid dates.
date_check = pd.to_datetime(
    df_clean[["year", "month", "day"]],
    errors="raise"
)

print()
print("Date range:", date_check.min().date(), "to", date_check.max().date())


Shape: (244, 15)
Missing values: 0
Duplicate rows: 0

Class counts:
Classes
fire        138
not fire    106
Name: count, dtype: int64

Region counts:
Region
Cordillera    122
Hudson Bay    122
Name: count, dtype: int64

Date range: 2012-06-01 to 2012-09-30


In [11]:
# Final structural checks.
assert df_clean.shape == (244, 15)
assert df_clean.isna().sum().sum() == 0
assert df_clean.duplicated().sum() == 0
assert set(df_clean["Classes"].unique()) == {"fire", "not fire"}
assert set(df_clean["Region"].unique()) == {"Cordillera", "Hudson Bay"}

print("All cleaning validation checks passed.")
df_clean.info()

All cleaning validation checks passed.
<class 'pandas.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   day          244 non-null    int64  
 1   month        244 non-null    int64  
 2   year         244 non-null    int64  
 3   Temperature  244 non-null    int64  
 4   RH           244 non-null    int64  
 5   Ws           244 non-null    int64  
 6   Rain         244 non-null    float64
 7   FFMC         244 non-null    float64
 8   DMC          244 non-null    float64
 9   DC           244 non-null    float64
 10  ISI          244 non-null    float64
 11  BUI          244 non-null    float64
 12  FWI          244 non-null    float64
 13  Classes      244 non-null    str    
 14  Region       244 non-null    str    
dtypes: float64(7), int64(6), str(2)
memory usage: 28.7 KB


## 7. Save the Cleaned Dataset

The clean file retains the original `day`, `month`, and `year` variables. Decisions about how to represent time are made in Notebook 02 during feature engineering.

In [12]:
df_clean.to_csv("forest_fires_clean.csv", index=False)
print("Saved: forest_fires_clean.csv")

Saved: forest_fires_clean.csv
